# AutoTablature - Audio Analysis and Model Training

This notebook demonstrates the audio analysis and model training process for the AutoTablature system.

In [ ]:
# Import necessary libraries
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
import torch.nn as nn
import json
from IPython.display import Audio
from pathlib import Path

# Add project root to path for imports
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Import project modules
from src.utils.config import Config
from src.audio.processor import AudioProcessor
from src.separation.separator import InstrumentSeparator
from src.models.pitch_detector import PitchDetector
from src.models.rhythm_analyzer import RhythmAnalyzer
from src.tablature.generator import TablatureGenerator

## 1. Load and Visualize Audio

In [ ]:
# Initialize configuration
config = Config()

# Initialize audio processor
audio_processor = AudioProcessor(config)

# Load a sample audio file
# Replace with your own audio file
sample_audio_path = os.path.join(project_root, 'data', 'samples', 'guitar_sample.mp3')

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(sample_audio_path), exist_ok=True)

# Check if file exists, display message if it doesn't
if not os.path.exists(sample_audio_path):
    print(f"Sample file {sample_audio_path} doesn't exist. Please add a sample audio file.")
    # You can download a sample file here...
else:
    # Load audio
    audio, sr = audio_processor.load_audio(sample_audio_path)
    
    # Display audio waveform
    plt.figure(figsize=(14, 5))
    librosa.display.waveshow(audio, sr=sr)
    plt.title('Audio Waveform')
    plt.tight_layout()
    plt.show()
    
    # Play audio
    Audio(data=audio, rate=sr)

## 2. Audio Feature Extraction

In [ ]:
# Preprocess audio
if 'audio' in locals():
    # Extract features
    features = audio_processor.preprocess((audio, sr))
    
    # Display spectrogram
    plt.figure(figsize=(14, 5))
    D = librosa.amplitude_to_db(np.abs(features['stft']), ref=np.max)
    librosa.display.specshow(D, y_axis='log', x_axis='time', sr=sr)
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram')
    plt.tight_layout()
    plt.show()
    
    # Display chromagram
    plt.figure(figsize=(14, 5))
    librosa.display.specshow(features['chroma'], y_axis='chroma', x_axis='time', sr=sr)
    plt.colorbar()
    plt.title('Chromagram')
    plt.tight_layout()
    plt.show()

## 3. Instrument Separation

In [ ]:
# Initialize separator
separator = InstrumentSeparator(config)

if 'features' in locals():
    # Separate instruments
    separated_tracks = separator.separate_instruments(features)
    
    # Display and play separated tracks
    for instrument, track_data in separated_tracks.items():
        print(f"Separated track: {instrument}")
        
        # Display waveform
        plt.figure(figsize=(14, 3))
        librosa.display.waveshow(track_data['audio'], sr=sr)
        plt.title(f'{instrument} Waveform')
        plt.tight_layout()
        plt.show()
        
        # Play audio
        display(Audio(data=track_data['audio'], rate=sr))

## 4. Pitch Detection

In [ ]:
# Initialize pitch detector
pitch_detector = PitchDetector(config)

if 'separated_tracks' in locals() and 'guitar_lead' in separated_tracks:
    # Detect pitches for lead guitar track
    guitar_track = separated_tracks['guitar_lead']
    pitches = pitch_detector.detect_pitches(guitar_track)
    
    # Display pitch detections
    plt.figure(figsize=(14, 5))
    
    # Convert MIDI notes to Hz for plotting
    notes = pitches['notes']
    times = pitches['timing']
    
    # Plot detected pitches
    plt.scatter(times, notes, alpha=0.8, s=15)
    plt.ylabel('MIDI Note')
    plt.xlabel('Time (s)')
    plt.title('Detected Pitches')
    plt.tight_layout()
    plt.show()
    
    # Display string/fret assignments
    string_assignments = pitches['string_assignments']
    
    for i in range(min(10, len(string_assignments))):
        print(f"Note {i+1}: Time={times[i]:.2f}s, MIDI={notes[i]:.1f}, " +
              f"String={string_assignments[i]['string']}, Fret={string_assignments[i]['fret']}")

## 5. Rhythm Analysis

In [ ]:
# Initialize rhythm analyzer
rhythm_analyzer = RhythmAnalyzer(config)

if 'separated_tracks' in locals() and 'guitar_lead' in separated_tracks:
    # Analyze rhythm for lead guitar track
    guitar_track = separated_tracks['guitar_lead']
    rhythm = rhythm_analyzer.analyze_rhythm(guitar_track)
    
    # Display rhythm analysis results
    print(f"Detected tempo: {rhythm['tempo']:.1f} BPM")
    print(f"Time signature: {rhythm['time_signature'][0]}/{rhythm['time_signature'][1]}")
    
    # Plot beat locations
    plt.figure(figsize=(14, 5))
    librosa.display.waveshow(guitar_track['audio'], sr=sr, alpha=0.6)
    
    # Plot beats
    for beat in rhythm['beats']:
        plt.axvline(x=beat, color='r', alpha=0.7, linestyle='--')
    
    # Plot onsets
    for onset in rhythm['onsets']:
        plt.axvline(x=onset, color='g', alpha=0.5)
        
    plt.title('Rhythm Analysis')
    plt.legend(['Waveform', 'Beats', 'Note Onsets'])
    plt.tight_layout()
    plt.show()

## 6. Tablature Generation

In [ ]:
# Initialize tablature generator
tablature_generator = TablatureGenerator(config)

if 'pitches' in locals() and 'rhythm' in locals():
    # Generate tablature for lead guitar track
    guitar_tablature = tablature_generator.generate_track(pitches, rhythm, instrument="guitar_lead")
    
    # Display tablature information
    print(f"Generated tablature for {guitar_tablature.instrument}")
    print(f"Number of measures: {len(guitar_tablature.measures)}")
    
    # Display first few measures
    for i, measure in enumerate(guitar_tablature.measures[:3]):
        print(f"\nMeasure {i+1}:")
        print(f"Time signature: {measure.time_signature[0]}/{measure.time_signature[1]}")
        print(f"Notes: {len(measure.notes)}")
        
        # Print ASCII tablature for this measure
        tab_lines = ["" for _ in range(6)]  # 6 strings
        
        for note in sorted(measure.notes, key=lambda n: n.beat_position):
            # Add spacing based on beat position
            for i in range(6):
                while len(tab_lines[i]) < int(note.beat_position * 8):
                    tab_lines[i] += "-"
            
            # Add note
            string_idx = note.string - 1
            fret_str = str(note.fret)
            
            for i in range(6):
                if i == string_idx:
                    tab_lines[i] += fret_str
                else:
                    tab_lines[i] += "-" * len(fret_str)
        
        # Print tablature
        for i, line in enumerate(tab_lines):
            print(f"{'EBGDAE'[i]}: |{line}|")

## 7. Model Training

In [ ]:
# Import trainer
from src.models.trainer import ModelTrainer

# Initialize trainer
trainer = ModelTrainer(config)

# Training data path
data_path = os.path.join(project_root, 'data', 'training')

# Check if training data exists
if not os.path.exists(data_path):
    print(f"Training data directory {data_path} doesn't exist.")
    print("Please create the directory and add training data.")
else:
    print("Training would start here if implemented...")
    # This would be a full training run
    # trainer.train(data_path, model_type="all")

## 8. Guitar Pro Output

In [ ]:
# Combine tracks into tablature document
if 'guitar_tablature' in locals():
    from src.tablature.generator import TablatureDocument
    
    # Create tablature document
    document = TablatureDocument()
    document.title = "AutoTablature Example"
    document.artist = "AI Generated"
    
    # Add tracks
    document.add_track(guitar_tablature)
    
    # Save as JSON (for now)
    output_path = os.path.join(project_root, 'data', 'output', 'example_tablature.json')
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    document.save(output_path)
    print(f"Saved tablature to {output_path}")